In [27]:
from src.envs import N3il

In [28]:
n = 5 # Example grid size, can be adjusted as needed
i = 0 # Random seed index, can be adjusted as needed

args = {
    'algorithm': 'MCTS',
    'n': n,
    'C': 1.41,  # 1e-7 for n=20
    'num_searches': 10*(n**2),  # Adjusted for larger n
    'num_workers': 1,      # >1 ⇒ parallel
    'virtual_loss': 1.0,     # magnitude to subtract at reservation
    'process_bar': True,
    'display_state': True,
    'logging_mode': False,
    'TopN': n,  # Without Priority
    "simulate_with_priority": False,
    'table_dir': f'tests/tests_mcts',  # Directory to save tables
    'figure_dir': f'tests/tests_mcts/figure',  # Directory to save figures
    'random_seed': i,  # Use the loop index as a seed for reproducibility
}

n3il_test = N3il((n,n), args=args)

In [29]:
state = n3il_test.get_initial_state()
state = n3il_test.get_next_state(state, 0)  # Example move
state

array([[1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]], dtype=uint8)

In [30]:
import numpy as np
from numba import njit

# D4 element codes (fixed enumeration)
E, R, R2, R3, SV, SH, SD, SA = 0, 1, 2, 3, 4, 5, 6, 7

# -------------------------
# Low-level coordinate maps
# -------------------------

@njit(cache=True, nogil=True)
def _map_coord(i, j, elem, row_count, col_count):
    """
    Map coordinates (i, j) under a D4 element 'elem' on an (row_count x col_count) grid.
    For non-square grids, only {E, R2, SV, SH} are meaningful; we never call others there.
    """
    if elem == E:   # identity
        return i, j
    elif elem == R:  # rotate 90° CCW (only valid for square)
        # (i, j) -> (j, n-1-i)
        return j, col_count - 1 - i
    elif elem == R2:  # rotate 180°
        # (i, j) -> (m-1-i, n-1-j)
        return row_count - 1 - i, col_count - 1 - j
    elif elem == R3:  # rotate 270° CCW (90° CW)
        # (i, j) -> (n-1-j, i)
        return row_count - 1 - j, i
    elif elem == SV:  # reflect vertical axis (left-right flip)
        # (i, j) -> (i, n-1-j)
        return i, col_count - 1 - j
    elif elem == SH:  # reflect horizontal axis (up-down flip)
        # (i, j) -> (m-1-i, j)
        return row_count - 1 - i, j
    elif elem == SD:  # reflect main diagonal y=x (only valid for square)
        # (i, j) -> (j, i)
        return j, i
    elif elem == SA:  # reflect anti-diagonal y=-x (only valid for square)
        # (i, j) -> (n-1-j, m-1-i)
        return col_count - 1 - j, row_count - 1 - i
    else:
        return i, j

@njit(cache=True, nogil=True)
def apply_element_to_action(action, elem, row_count, col_count):
    """Apply a D4 element to a flattened action index."""
    i = action // col_count
    j = action % col_count
    ni, nj = _map_coord(i, j, elem, row_count, col_count)
    return ni * col_count + nj

# --------------------------------
# Detect stabilizer subgroup G_x
# --------------------------------

@njit(cache=True, nogil=True)
def _element_fixes_state(elem, state):
    """
    Check if D4 element 'elem' fixes 'state' pointwise.
    We compare state[i,j] with state[ mapped(i,j) ] for all cells.
    """
    m, n = state.shape
    for i in range(m):
        for j in range(n):
            ii, jj = _map_coord(i, j, elem, m, n)
            if state[i, j] != state[ii, jj]:
                return False
    return True

@njit(cache=True, nogil=True)
def detect_stabilizer_elements_nb(state):
    """
    Return an 8-length boolean array 'fix' where fix[elem]=True
    iff the D4 element 'elem' fixes the state.

    For non-square grids, we skip checks for R, R3, SD, SA (set to False).
    """
    m, n = state.shape
    square = (m == n)

    fix = np.zeros(8, dtype=np.bool_)
    # Always consider E, R2, SV, SH
    fix[E]  = _element_fixes_state(E,  state)
    fix[R2] = _element_fixes_state(R2, state)
    fix[SV] = _element_fixes_state(SV, state)
    fix[SH] = _element_fixes_state(SH, state)

    if square:
        fix[R]  = _element_fixes_state(R,  state)
        fix[R3] = _element_fixes_state(R3, state)
        fix[SD] = _element_fixes_state(SD, state)
        fix[SA] = _element_fixes_state(SA, state)
    else:
        fix[R] = False
        fix[R3] = False
        fix[SD] = False
        fix[SA] = False

    return fix

# ----------------------------------------------------
# Match stabilizer to one of the 10 canonical subgroups
# ----------------------------------------------------

@njit(cache=True, nogil=True)
def _fill_row(row, elems):
    """
    Helper: write a subgroup's element indices into a row (length 8),
    fill unused slots with -1.
    """
    for k in range(8):
        row[k] = -1
    for k in range(len(elems)):
        row[k] = elems[k]

@njit(cache=True, nogil=True)
def _build_canonical_subgroups():
    """
    Returns:
      subs (10 x 8 int array): each row lists the element indices of a canonical subgroup, -1 padded
      sizes (10,): number of elements in each subgroup row
      ids (10,): arbitrary IDs 0..9 for reference
        0:{e}, 1:<r>, 2:<r^2>, 3:<s_v>, 4:<s_h>, 5:<s_d>, 6:<s_a>, 7:V1, 8:V2, 9:D4
    """
    subs = np.empty((10, 8), dtype=np.int64)
    sizes = np.empty(10, dtype=np.int64)
    ids = np.arange(10, dtype=np.int64)

    # 0: {e}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E]))
    subs[0] = row; sizes[0] = 1

    # 1: <r> = {e, r, r2, r3}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, R, R2, R3]))
    subs[1] = row; sizes[1] = 4

    # 2: <r^2> = {e, r2}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, R2]))
    subs[2] = row; sizes[2] = 2

    # 3: <s_v> = {e, s_v}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, SV]))
    subs[3] = row; sizes[3] = 2

    # 4: <s_h> = {e, s_h}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, SH]))
    subs[4] = row; sizes[4] = 2

    # 5: <s_d> = {e, s_d}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, SD]))
    subs[5] = row; sizes[5] = 2

    # 6: <s_a> = {e, s_a}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, SA]))
    subs[6] = row; sizes[6] = 2

    # 7: V1 = {e, r2, s_v, s_h}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, R2, SV, SH]))
    subs[7] = row; sizes[7] = 4

    # 8: V2 = {e, r2, s_d, s_a}
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, R2, SD, SA]))
    subs[8] = row; sizes[8] = 4

    # 9: D4 (all eight)
    row = np.empty(8, dtype=np.int64); _fill_row(row, np.array([E, R, R2, R3, SV, SH, SD, SA]))
    subs[9] = row; sizes[9] = 8

    return subs, sizes, ids

@njit(cache=True, nogil=True)
def _fixes_equals_subgroup(fix, subs_row):
    """
    Check whether the boolean 'fix' set equals the subgroup listed in 'subs_row'.
    """
    listed = np.zeros(8, dtype=np.bool_)
    for k in range(8):
        idx = subs_row[k]
        if idx == -1:
            break
        listed[idx] = True

    # exact equality
    for e in range(8):
        if fix[e] != listed[e]:
            return False
    return True

@njit(cache=True, nogil=True)
def identify_stabilizer_subgroup_nb(state):
    """
    Detect the stabilizer elements, then match exactly to one of the 10 canonical subgroups.
    Returns:
      subgroup_id (0..9 as documented above),
      subgroup_elems (length <= 8, filled with -1 beyond size),
      subgroup_size
    If no exact match (shouldn't happen), fall back to the literal 'fix' set.
    """
    fix = detect_stabilizer_elements_nb(state)
    subs, sizes, ids = _build_canonical_subgroups()

    # Try to match exactly one canonical subgroup
    for r in range(10):
        if _fixes_equals_subgroup(fix, subs[r]):
            return ids[r], subs[r], sizes[r]

    # Fallback: construct subgroup row directly from 'fix'
    # (This would be unusual; included for robustness.)
    tmp = np.empty(8, dtype=np.int64)
    cnt = 0
    for e in range(8):
        if fix[e]:
            tmp[cnt] = e
            cnt += 1
    for k in range(cnt, 8):
        tmp[k] = -1
    return -1, tmp, cnt  # id -1 = non-canonical (should not occur)

# -----------------------------------------
# Subgroup-based symmetric action filtering
# -----------------------------------------

@njit(cache=True, nogil=True)
def filter_actions_by_stabilizer_nb(valid_moves, state, row_count, col_count):
    """
    Reduce action space by orbits under the stabilizer subgroup G_x of the current state.
    Keep the minimum flattened index in each orbit.

    Args:
      valid_moves: 1D boolean array
      state: 2D uint8 array
      row_count, col_count: ints

    Returns:
      filtered_moves: 1D boolean array
    """
    # Identify the stabilizer subgroup (one of the 10)
    subgroup_id, subgroup_row, subgroup_size = identify_stabilizer_subgroup_nb(state)

    # If stabilizer is trivial {e}, return original
    if subgroup_size <= 1:
        return valid_moves

    filtered = valid_moves.copy()
    N = valid_moves.shape[0]

    # Pre-extract subgroup elements into a compact array
    elems = np.empty(subgroup_size, dtype=np.int64)
    for k in range(subgroup_size):
        elems[k] = subgroup_row[k]  # no -1 within subgroup_size

    # Iterate over valid indices; for each orbit, keep the minimal index
    idxs = np.where(valid_moves)[0]
    for t in range(idxs.shape[0]):
        a = idxs[t]
        if not filtered[a]:
            continue

        # Build orbit under G_x
        min_a = a
        orbit = np.empty(subgroup_size, dtype=np.int64)
        for k in range(subgroup_size):
            b = apply_element_to_action(a, elems[k], row_count, col_count)
            orbit[k] = b
            if b < min_a:
                min_a = b

        # Disable non-canonical members (keep only min_a)
        for k in range(subgroup_size):
            b = orbit[k]
            if b != min_a and b < N:
                filtered[b] = False

    return filtered

# -----------------------------
# Class integration (override)
# -----------------------------

class N3il_with_symmetry(N3il):
    """
    N3il enhanced with subgroup-based action filtering.
    Filtering is done by the state stabilizer subgroup G_x (one of the 10 subgroups of D4 on squares).
    """

    def __init__(self, grid_size, args, priority_grid=None):
        super().__init__(grid_size, args, priority_grid)

    def get_valid_moves(self, state):
        # Parent valid moves (already possibly TopN-prioritized)
        valid_moves = super().get_valid_moves(state)
        # Subgroup-based filtering
        return filter_actions_by_stabilizer_nb(
            valid_moves, state, self.row_count, self.column_count
        )

    def get_valid_moves_subset(self, parent_state, parent_valid_moves, action_taken):
        valid_moves = super().get_valid_moves_subset(parent_state, parent_valid_moves, action_taken)

        # Build child state to compute stabilizer at the node where these moves apply
        child_state = parent_state.copy()
        r = action_taken // self.column_count
        c = action_taken % self.column_count
        child_state[r, c] = 1

        return filter_actions_by_stabilizer_nb(
            valid_moves, child_state, self.row_count, self.column_count
        )

In [46]:
# All examples are 5x5 grids for consistency
size = 5

# 0: G = {E}
state_e = np.array([[1, 1, 0, 0, 0], 
                    [0, 0, 0, 0, 0], 
                    [0, 0, 0, 0, 0], 
                    [0, 0, 0, 0, 0], 
                    [0, 0, 0, 0, 0]], dtype=np.uint8)

# 1: G = <R> = {E, R, R2, R3}
state_r = np.array([[0, 0, 0, 1, 0], 
                    [1, 0, 0, 0, 0], 
                    [0, 0, 0, 0, 0], 
                    [0, 0, 0, 0, 1], 
                    [0, 1, 0, 0, 0]], dtype=np.uint8)

# 2: G = <R2> = {E, R2}
state_r2 = np.array([[0, 0, 0, 1, 1], 
                     [0, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0], 
                     [1, 1, 0, 0, 0]], dtype=np.uint8)

# 3: G = <SV> = {E, SV}
state_sv = np.array([[0, 1, 0, 1, 0], 
                     [0, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0]], dtype=np.uint8)

# 4: G = <SH> = {E, SH}
state_sh = np.array([[0, 0, 0, 0, 0], 
                     [1, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0], 
                     [1, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0]], dtype=np.uint8)

# 5: G = <SD> = {E, SD}
state_sd = np.array([[0, 1, 0, 0, 0], 
                     [1, 0, 0, 0, 0], 
                     [0, 0, 0, 0, 0], 
                     [0, 0, 0, 1, 0], 
                     [0, 0, 0, 0, 0]], dtype=np.uint8)

# 6: G = <SA> = {E, SA}
state_sa = np.array([[0, 0, 0, 1, 0], 
                     [0, 0, 0, 0, 1], 
                     [0, 0, 0, 0, 0], 
                     [0, 1, 0, 0, 0], 
                     [0, 0, 0, 0, 0]], dtype=np.uint8)

# 7: G = V1 = {E, R2, SV, SH}
state_v1 = np.array([[0, 0, 0, 0, 0], 
                     [1, 0, 0, 0, 1], 
                     [0, 0, 1, 0, 0], 
                     [1, 0, 0, 0, 1], 
                     [0, 0, 0, 0, 0]], dtype=np.uint8)

# 8: G = V2 = {E, R2, SD, SA}
state_v2 = np.array([[0, 0, 0, 1, 0], 
                     [0, 0, 0, 0, 1], 
                     [0, 0, 0, 0, 0], 
                     [1, 0, 0, 0, 0], 
                     [0, 1, 0, 0, 0]], dtype=np.uint8)

# 9: G = D4 (all eight elements)
state_d4 = np.array([[1, 0, 0, 0, 1], 
                     [0, 0, 0, 0, 0], 
                     [0, 0, 1, 0, 0], 
                     [0, 0, 0, 0, 0], 
                     [1, 0, 0, 0, 1]], dtype=np.uint8)

# --- Instantiate the class and print the results ---
n_sym = N3il_with_symmetry((size, size), args=args)

print("State 0: G = {E}")
print(state_e)
print("Valid moves:")
print(n_sym.get_valid_moves(state_e).reshape((size, size)))

print("\nState 1: G = <R>")
print(state_r)
print("Valid moves:")
print(n_sym.get_valid_moves(state_r).reshape((size, size)))

print("\nState 2: G = <R2>")
print(state_r2)
print("Valid moves:")
print(n_sym.get_valid_moves(state_r2).reshape((size, size)))

print("\nState 3: G = <SV>")
print(state_sv)
print("Valid moves:")
print(n_sym.get_valid_moves(state_sv).reshape((size, size)))

print("\nState 4: G = <SH>")
print(state_sh)
print("Valid moves:")
print(n_sym.get_valid_moves(state_sh).reshape((size, size)))

print("\nState 5: G = <SD>")
print(state_sd)
print("Valid moves:")
print(n_sym.get_valid_moves(state_sd).reshape((size, size)))

print("\nState 6: G = <SA>")
print(state_sa)
print("Valid moves:")
print(n_sym.get_valid_moves(state_sa).reshape((size, size)))

print("\nState 7: G = V1")
print(state_v1)
print("Valid moves:")
print(n_sym.get_valid_moves(state_v1).reshape((size, size)))

print("\nState 8: G = V2")
print(state_v2)
print("Valid moves:")
print(n_sym.get_valid_moves(state_v2).reshape((size, size)))

print("\nState 9: G = D4")
print(state_d4)
print("Valid moves:")
print(n_sym.get_valid_moves(state_d4).reshape((size, size)))

State 0: G = {E}
[[1 1 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves:
[[0 0 0 0 0]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]]

State 1: G = <R>
[[0 0 0 1 0]
 [1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 1]
 [0 1 0 0 0]]
Valid moves:
[[1 1 1 0 0]
 [0 1 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

State 2: G = <R2>
[[0 0 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 0 0]]
Valid moves:
[[0 0 0 0 0]
 [1 1 1 0 1]
 [1 1 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

State 3: G = <SV>
[[0 1 0 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves:
[[0 0 0 0 0]
 [1 1 1 0 0]
 [1 1 1 0 0]
 [1 1 1 0 0]
 [1 1 1 0 0]]

State 4: G = <SH>
[[0 0 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 1 1 1 1]
 [0 1 1 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

State 5: G = <SD>
[[0 1 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[1 0 1 1 1]
 [0 1 1 1 1]
 [0 0 1 1 1]
 [0 0 0 0 1]
 [0 0 0 0 1]]

State 6: G = <SA>


In [ ]:
c

In [34]:
n3il_with_sym.get_valid_moves(state).reshape((n, n))  # Display valid moves in grid format

array([[0, 1, 1, 1, 1],
       [0, 1, 1, 1, 1],
       [0, 0, 1, 1, 1],
       [0, 0, 0, 1, 1],
       [0, 0, 0, 0, 1]], dtype=uint8)

In [35]:
n3il_with_sym.get_next_state(state, 1)  # Example move to see the next state

array([[1, 1, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]], dtype=uint8)

In [36]:
n3il_with_sym.get_valid_moves(state).reshape((n,n))

array([[0, 0, 0, 0, 0],
       [1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1]], dtype=uint8)

In [37]:
n3il_with_sym.get_next_state(state, 20)
n3il_with_sym.get_next_state(state, 21)
state

array([[1, 1, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [1, 1, 0, 0, 0]], dtype=uint8)

In [38]:
n3il_with_sym.get_valid_moves(state).reshape((n,n))

array([[0, 0, 0, 0, 0],
       [0, 0, 1, 1, 1],
       [0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]], dtype=uint8)

In [21]:
# Define all symmetric types of the current state
def get_all_symmetries(state):
    syms = []
    syms.append(state)  # original
    syms.append(np.flipud(state))  # horizontal
    syms.append(np.fliplr(state))  # vertical
    syms.append(np.flip(state, (0, 1)))  # central (180 rotation)
    if state.shape[0] == state.shape[1]:
        syms.append(state.T)  # main diagonal
        syms.append(np.flip(state.T, (0, 1)))  # anti-diagonal
    return syms

sym_names = [
    'original',
    'horizontal',
    'vertical',
    'central',
    'main_diagonal',
    'anti_diagonal'
]

all_syms = get_all_symmetries(state)
for idx, sym_state in enumerate(all_syms):
    print(f"\nSymmetry: {sym_names[idx]}")
    print(sym_state)
    print("Valid moves after symmetry filtering:")
    print(n3il_with_sym.get_valid_moves(sym_state).reshape((5,5)))


Symmetry: original
[[1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves after symmetry filtering:
[[0 1 1 1 1]
 [0 1 1 1 1]
 [0 0 1 1 1]
 [0 0 0 1 1]
 [0 0 0 0 1]]

Symmetry: horizontal
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 0 0 0 0]]
Valid moves after symmetry filtering:
[[1 1 1 1 1]
 [1 1 1 1 0]
 [1 1 1 0 0]
 [1 1 0 0 0]
 [0 0 0 0 0]]

Symmetry: vertical
[[0 0 0 0 1]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves after symmetry filtering:
[[1 1 1 1 0]
 [1 1 1 1 0]
 [1 1 1 0 0]
 [1 1 0 0 0]
 [1 0 0 0 0]]

Symmetry: central
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 1]]
Valid moves after symmetry filtering:
[[1 1 1 1 1]
 [0 1 1 1 1]
 [0 0 1 1 1]
 [0 0 0 1 1]
 [0 0 0 0 0]]

Symmetry: main_diagonal
[[1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves after symmetry filtering:
[[1 1 1 1 1]
 [1 1 1 1 0]
 [1 1 1 0 0]
 [1 1 0 0 0]
 [0 0 0 0 0]]

Symmetry: vertical
[[0 0 0 0 1]
 [0 0 0 0 0

In [22]:
import numpy as np

# Define 8 distinct 5x5 states, each representing a D4 symmetry type
def make_example_states():
    states = []
    # 1. Identity (original)
    s0 = np.zeros((5,5), dtype=np.uint8)
    s0[1, 1] = 1; s0[2, 2] = 1; s0[3, 3] = 1
    states.append(s0)

    # 2. Horizontal flip
    s1 = np.flipud(s0)
    states.append(s1)

    # 3. Vertical flip
    s2 = np.fliplr(s0)
    states.append(s2)

    # 4. 180-degree rotation (central)
    s3 = np.flip(s0, (0, 1))
    states.append(s3)

    # 5. Main diagonal (transpose)
    s4 = s0.T
    states.append(s4)

    # 6. Anti-diagonal
    s5 = np.flip(s0.T, (0, 1))
    states.append(s5)

    # 7. Rotate 90 degrees (clockwise)
    s6 = np.rot90(s0, 1)
    states.append(s6)

    # 8. Rotate 270 degrees (counterclockwise)
    s7 = np.rot90(s0, 3)
    states.append(s7)

    return states

sym_names = [
    'identity',
    'horizontal',
    'vertical',
    'central',
    'main_diagonal',
    'anti_diagonal',
    'rotate_90',
    'rotate_270'
]

example_states = make_example_states()

for idx, s in enumerate(example_states):
    print(f'\nSymmetry type: {sym_names[idx]}')
    print(s)
    print('Valid moves:')
    print(n3il_with_sym.get_valid_moves(s).reshape((5,5)))


Symmetry type: identity
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: horizontal
[[0 0 0 0 0]
 [0 0 0 1 0]
 [0 0 1 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]]
Valid moves:
[[1 1 1 1 0]
 [0 1 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: vertical
[[0 0 0 0 0]
 [0 0 0 1 0]
 [0 0 1 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]]
Valid moves:
[[1 1 1 1 0]
 [0 1 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: central
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: main_diagonal
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: anti_diagonal
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0